# Jev_CeNN — Two Sparse Attention Windows + Effective CeNN: Compress Layers 15–17 into ONE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Jev_CeNN_2Window_3Layers_To_1_Colab.ipynb)

## Research objective

**Replace the complete original ModernBERT layer sequence 15 (global), 16 (sliding), 17 (sliding), including attention, residual norms and THREE feedforward sublayers, with ONE trainable Jev-CeNN superlayer containing two attention stages, ONE feedforward sublayer, and an active cellular pathway.**

This is not merely swapping three attention modules. `encoder.layers[15]` becomes the superlayer, and `encoder.layers[16]` and `encoder.layers[17]` become parameter-free identity/bypass layers with the original ModernBERT tuple forward contract.

**Stage 1 — train CeNN to be useful:** distill a shared three-scale cellular mixer (receptive fields 7,33,129) on original teacher layer-15 residual. Measure whether a purely cellular pathway can reconstruct residual features.

**Stage 2 — global coverage with two windows:** local W45 radius 22; dilated W45 radius 22 at stride 23. Every token at T≤512 has a two-hop path to every other token, but this does NOT prove arbitrary teacher attention matrices can be exactly factored.

**Stage 3 — 3→1 segment distillation:** original teacher layer-15 *input* → original teacher layer-17 *output*, with an intermediate loss on teacher layer-15 output. Only the student superlayer trains. All other layers and task head remain teacher-identical and frozen. Both QK attention stages remain query-dependent and differentiable.

**Stage 4 — test whether CeNN is actually effective:** disable ALL CeNN computations in the trained block **without retraining**. Compare local block fidelity, downstream teacher agreement, task gold accuracy, CUDA latency; optionally train an equal-budget attention-only control. A CeNN that has no measurable held-out benefit is reported as not proven useful.

**Stage 5 — deployment gate:** exactly 3-to-1 layer replacement is tested inside the whole encoder. A usable checkpoint requires **audit cosine ≥0.99, NMSE ≤0.025, DEV teacher agreement ≥0.99, and no DEV gold-accuracy drop >1.5 percentage points**. Performance against native teacher is reported separately; the model is not called faster unless measured.

Only the typed-decisions original TRAIN split is used, divided by case into TRAIN / PROBE / AUDIT / downstream DEV. No official TEST, no Hugging Face publication, and no changes to the existing verified `vtava/Jev_CeNN`.

## Mathematics and hypothesis

With first attention radius R1=22, second attention radius R2=22 at stride S=23,

\[
Z_i = \sum_{j\in[i-22,i+22]} \alpha_{ij}V_j + \gamma_1\operatorname{CeNN}(X)_i + X_i,
\qquad
Y_i = Z_i + \sum_{m=-22}^{22}\beta_{i,m}V'_{i+23m}+\gamma_2\operatorname{CeNN}(Z)_i +\operatorname{FFN}(\cdot)_i
\]

Boundary/padding masks apply. Nominal attention scores at T=512: `512 × (45 + 45) = 46,080`, compared with `512² = 262,144` in one dense teacher attention module. **Three teacher layers**, projections, CeNN convolutions, and data movement must be included in *actual measured* GPU-speed claims.

The three-to-one mapping is a hypothesis: full information *reachability* is not the same as representational equivalence or guaranteed gradient convergence.

**Outputs:** JSON report, probe-training histories, reusable checkpoint (`ACCEPTED` or `EXPERIMENTAL_NOT_DEPLOYABLE`), and Pareto plots.


In [ ]:
#@title 1. Pinned Transformers 4.57.6 and Jev-CeNN dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20,<5" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Frozen original teacher, strict 15-17 geometry, and Colab configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv, _valid_tokens
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
)
from tinycenn_lm.decision_training import require_finite, decision_distill_loss, guarded_step
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion

SEED=2028
SOURCE_MODEL="convaiinnovations/laya-typed-decisions" #@param {type:"string"}
FIRST_LAYER=15
LAST_LAYER=17
MAX_LEN=512 #@param {type:"integer"}
LOCAL_RADIUS=22 #@param {type:"integer"}
DILATION_STRIDE=23 #@param {type:"integer"}
DILATED_RADIUS=22 #@param {type:"integer"}
TRAIN_CASES=600 #@param {type:"integer"}
DEV_CASES=180 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES=40 #@param {type:"integer"}
PROBE_CACHE_BATCHES=12 #@param {type:"integer"}
AUDIT_CACHE_BATCHES=12 #@param {type:"integer"}
CENN_STEPS=180 #@param {type:"integer"}
SUPERLAYER_STEPS=400 #@param {type:"integer"}
ATTENTION_ONLY_RECOVERY_STEPS=0 #@param {type:"integer"}
QUALITY_DEV_LIMIT=240 #@param {type:"integer"}
CHECK_EVERY=40
CENN_AUX_WEIGHT=0.05 #@param {type:"number"}
INTERMEDIATE_WEIGHT=0.15 #@param {type:"number"}
TARGET_COSINE=0.99
TARGET_NMSE=0.025
TARGET_AGREEMENT=0.99
MAX_GOLD_DROP=0.015
OUTPUT_DIR=Path("/content/Jev_CeNN_TwoWindow_15to17")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert FIRST_LAYER==15 and LAST_LAYER==17
assert BATCH>=1 and CENN_STEPS>=1 and SUPERLAYER_STEPS>=1
assert ATTENTION_ONLY_RECOVERY_STEPS>=0 and QUALITY_DEV_LIMIT>=64
assert MAX_LEN>=128
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU (Colab > Runtime > Change runtime type).")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL,
    allow_patterns=["rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*"],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

print("GPU:", torch.cuda.get_device_name(0), "| AMP:", amp_dtype)
from tinycenn_lm.standalone_decision import full_attention_indices
GLOBAL_LAYERS = full_attention_indices(teacher)
SLIDING_LAYERS = [i for i in range(len(teacher.encoder.layers)) if i not in GLOBAL_LAYERS]
assert len(teacher.encoder.layers)==28 and [i for i in (15,16,17) if i in GLOBAL_LAYERS]==[15]
for i in GLOBAL_LAYERS:
    assert tuple(teacher.encoder.layers[i].attn.local_attention) == (-1, -1)
print("Global layers:", GLOBAL_LAYERS, "| native sliding:", SLIDING_LAYERS)
print("Two attention windows:",2*LOCAL_RADIUS+1,"and",2*DILATED_RADIUS+1, "stride",DILATION_STRIDE,"| MAX_LEN:",MAX_LEN)
assert all(tuple(teacher.encoder.layers[i].attn.local_attention)==(64,64) for i in (16,17))
print("Original 3 complete blocks will become one trainable superlayer and 2 parameter-free bypasses.")
print("Teacher:", SOURCE_MODEL, "| official test split stays untouched")


In [ ]:
#@title 3. Case-disjoint train/probe/audit/DEV original teacher data
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, True)

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
        # ModernBERT SDPA requires position_ids and sliding_window_mask.
        # Replay the exact teacher call, including all positional/mask kwargs.
        got["call_kwargs"] = {
            key: (value.detach() if torch.is_tensor(value) else value)
            for key, value in kwargs.items()
        }
        if "position_ids" not in got["call_kwargs"] or "sliding_window_mask" not in got["call_kwargs"]:
            raise RuntimeError("Teacher attention hook did not capture ModernBERT SDPA kwargs")
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred = repl(entry["x"], **entry["call_kwargs"])[0]
            require_finite(pred, "all-attention local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/dev decisions:", len(train_items), len(dev_items))
assert len(dev_items) >= BATCH * (PROBE_CACHE_BATCHES + AUDIT_CACHE_BATCHES), "Increase DEV_CASES for disjoint probe and audit caches"
print("official test remains untouched; probe/audit I/O non-overlapping")


In [ ]:
#@title 4. 512-token global reachability proof and original 3-block instrumentation
from tinycenn_lm.jev_cenn_pairwindow_superlayer import (
    JevCeNNPairWindowSuperLayer, BypassModernBertLayer,
    install_superlayer, restore_original_layers,
    pair_reachable, pair_coverage_fraction
)
assert pair_reachable(MAX_LEN,LOCAL_RADIUS,DILATION_STRIDE,DILATED_RADIUS), (
    "Chosen geometry cannot reach every other token in two hops!")
for length in (17,45,129,256,MAX_LEN):
    fraction=pair_coverage_fraction(
        length,LOCAL_RADIUS,DILATION_STRIDE,DILATED_RADIUS)
    assert fraction==1.0,(length,fraction)
print("Two-hop reachability verified through T=",MAX_LEN,
      "| scores/sequence at T512:",
      MAX_LEN*(2*LOCAL_RADIUS+1+2*DILATED_RADIUS+1),
      "| native global pairs:",MAX_LEN*MAX_LEN)
superlayer=JevCeNNPairWindowSuperLayer(
    teacher.encoder.layers[15],teacher.encoder.layers[16],
    local_radius=LOCAL_RADIUS,stride=DILATION_STRIDE,
    dilated_radius=DILATED_RADIUS,max_len=MAX_LEN,
).to(device)
assert len(teacher.encoder.layers)==len(student_eval.encoder.layers)==28
print("Trainable superlayer parameters:",
      sum(p.numel() for p in superlayer.parameters())/1e6,"M")


In [ ]:
#@title 5. Capture original teacher complete layer 15 input -> layer 17 output
# Original three layers remain untouched in teacher. Cache each train/probe/
# blind audit batch using ONE teacher forward and three lightweight hooks.
def _cpu_detached(value):
    if torch.is_tensor(value):
        v=value.detach().cpu()
        return v.half().contiguous() if v.is_floating_point() else v.contiguous()
    return value
def capture_three(items,count,offset,label):
    entries=[]
    for n in range(count):
        b=batch(items,offset+n*BATCH)
        found={}
        def pre(mod,args,kwargs):
            found["input"]=args[0].detach()
            found["position_ids"]=kwargs.get("position_ids")
        def last(index):
            def post(mod,args,kwargs,outputs):
                found[f"y{index}"]=outputs[0].detach()
            return post
        layers=teacher.encoder.layers
        hooks=[
            layers[15].register_forward_pre_hook(pre,with_kwargs=True),
            layers[15].register_forward_hook(last(15),with_kwargs=True),
            layers[16].register_forward_hook(last(16),with_kwargs=True),
            layers[17].register_forward_hook(last(17),with_kwargs=True),
        ]
        try:
            with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
                teacher(**b)
        finally:
            for h in hooks:h.remove()
        assert all(key in found for key in ("input","y15","y16","y17"))
        e={"x":_cpu_detached(found["input"]),
           "y15":_cpu_detached(found["y15"]),
           "y16":_cpu_detached(found["y16"]),
           "y17":_cpu_detached(found["y17"]),
           "valid":b["attention_mask"].bool().cpu(),
           "position_ids":_cpu_detached(found["position_ids"])}
        entries.append(e)
        if (n+1)%8==0 or n+1==count:
            print(f"{label} teacher segment batches {n+1}/{count}")
    return entries
TRAIN_CACHE=capture_three(train_items,TRAIN_CACHE_BATCHES,0,"TRAIN")
PROBE_CACHE=capture_three(dev_items,PROBE_CACHE_BATCHES,0,"PROBE")
AUDIT_CACHE=capture_three(
    dev_items,AUDIT_CACHE_BATCHES,PROBE_CACHE_BATCHES*BATCH,"AUDIT")
DEV_QUALITY=dev_items[(PROBE_CACHE_BATCHES+AUDIT_CACHE_BATCHES)*BATCH:]
assert len(DEV_QUALITY)>=QUALITY_DEV_LIMIT, (
    "Insufficient independent DEV cases; increase DEV_CASES.")
def gpu_entry(e):
    return {k:(v.to(device,non_blocking=True) if torch.is_tensor(v) else v)
            for k,v in e.items()}
print("Three-block teacher reference cached. Train/probe/audit/DEV disjoint.")


In [ ]:
#@title 6. Stage A: train the CeNN alone on original layer-15 residual
def metric_and_loss(pred,target,valid):
    mask=valid[:,:,None].float()
    p=pred.float()
    y=target.float()
    mse=((p-y).square()*mask).sum()
    power=(y.square()*mask).sum().clamp_min(1e-8)
    nmse=mse/power
    cosine=F.cosine_similarity(p[valid],y[valid],dim=-1).mean()
    return nmse+1.5*(1-cosine),nmse,cosine
def cpu_snapshot(module):
    return {k:v.detach().float().cpu().clone()
            for k,v in module.state_dict().items()}
def probe_metrics(model,cache,kind="final"):
    model.eval()
    diff=power=cos_sum=0.0
    n_tokens=0
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        for entry in cache:
            e=gpu_entry(entry)
            if kind=="cenn":
                predicted=model.cellular_only_prediction(e["x"],e["valid"])
                target=e["y15"]
            else:
                predicted=model(
                    e["x"],attention_mask=e["valid"],
                    position_ids=e["position_ids"])[0]
                target=e["y17"]
            valid=e["valid"]
            m=valid[:,:,None].float()
            diff+=float(((predicted.float()-target.float()).square()*m).sum())
            power+=float((target.float().square()*m).sum())
            cos_sum+=float(F.cosine_similarity(
                predicted.float()[valid],target.float()[valid],dim=-1).sum())
            n_tokens+=int(valid.sum())
    return {"cosine":cos_sum/max(n_tokens,1),
            "nmse":diff/max(power,1e-8),
            "tokens":n_tokens}
stage_a_opt=torch.optim.AdamW(superlayer.cenn.parameters(),
                               lr=0.0008,weight_decay=0.001)
scaler_a=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
best_a=probe_metrics(superlayer,PROBE_CACHE,kind="cenn")
best_a_score=best_a["nmse"]+1.5*(1-best_a["cosine"])
best_a_state=cpu_snapshot(superlayer.cenn)
history_a=[{"step":0,**best_a}]
updates=attempts=0
while updates<CENN_STEPS:
    if attempts>CENN_STEPS+300:
        raise RuntimeError("Stage A too many skipped AMP updates")
    e=gpu_entry(TRAIN_CACHE[attempts%len(TRAIN_CACHE)])
    superlayer.cenn.train()
    stage_a_opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        predicted=superlayer.cellular_only_prediction(e["x"],e["valid"])
        loss,_,_=metric_and_loss(predicted,e["y15"],e["valid"])
    require_finite(loss,"CeNN stage A")
    scaler_a.scale(loss).backward()
    scaler_a.unscale_(stage_a_opt)
    torch.nn.utils.clip_grad_norm_(superlayer.cenn.parameters(),1.0)
    scale=scaler_a.get_scale()
    scaler_a.step(stage_a_opt)
    scaler_a.update()
    attempts+=1
    if scaler_a.is_enabled() and scaler_a.get_scale()<scale:
        continue
    updates+=1
    if updates==1 or updates%CHECK_EVERY==0 or updates==CENN_STEPS:
        current=probe_metrics(superlayer,PROBE_CACHE,kind="cenn")
        history_a.append({"step":updates,**current})
        rank=current["nmse"]+1.5*(1-current["cosine"])
        if rank<best_a_score:
            best_a=current;best_a_score=rank
            best_a_state=cpu_snapshot(superlayer.cenn)
        print(f"CeNN Stage A {updates:4d}/{CENN_STEPS}: "
              f"probe cosine={current['cosine']:.6f} NMSE={current['nmse']:.5f}")
superlayer.cenn.load_state_dict(best_a_state,strict=True)
print("Best CeNN-only reconstruction on probe:",best_a)
del stage_a_opt,scaler_a,best_a_state
torch.cuda.empty_cache()


In [ ]:
#@title 7. Stage B: train ONE two-window-CeNN block against the original THREE blocks
# All original teacher and other student layers remain frozen. The first
# local path is also supervised against the original layer-15 output.
for p in superlayer.parameters():p.requires_grad_(True)
base_lr=4e-4
optimizer=torch.optim.AdamW([
    {"params":[p for n,p in superlayer.named_parameters()
               if not n.startswith(("qkv1.","qkv2."))],
     "lr":base_lr,"base_lr":base_lr},
    {"params":[p for n,p in superlayer.named_parameters()
               if n.startswith(("qkv1.","qkv2."))],
     "lr":2e-5,"base_lr":2e-5},
],weight_decay=2e-4)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
best=probe_metrics(superlayer,PROBE_CACHE)
best_rank=best["nmse"]+1.5*(1-best["cosine"])
best_state=cpu_snapshot(superlayer)
best_step=0
history=[{"step":0,**best}]
updates=attempts=0
while updates<SUPERLAYER_STEPS:
    if attempts>SUPERLAYER_STEPS+400:
        raise RuntimeError("Stage B too many AMP overflows")
    e=gpu_entry(TRAIN_CACHE[attempts%len(TRAIN_CACHE)])
    superlayer.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        output=superlayer(
            e["x"],attention_mask=e["valid"],
            position_ids=e["position_ids"])[0]
        main,_,_=metric_and_loss(output,e["y17"],e["valid"])
        intermediate,_,_=metric_and_loss(
            superlayer.last_mid,e["y15"],e["valid"])
        cenn_proxy=superlayer.cellular_only_prediction(
            e["x"],e["valid"])
        auxiliary,_,_=metric_and_loss(cenn_proxy,e["y15"],e["valid"])
        loss=(main+INTERMEDIATE_WEIGHT*intermediate+
              CENN_AUX_WEIGHT*auxiliary)
    require_finite(loss,"Jev_CeNN two-window superlayer loss")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(superlayer.parameters(),1.0)
    progress=(updates+1)/SUPERLAYER_STEPS
    for group in optimizer.param_groups:
        group["lr"]=group["base_lr"]*(
            0.15+0.85*0.5*(1+math.cos(math.pi*progress)))
    scale=scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    attempts+=1
    if scaler.is_enabled() and scaler.get_scale()<scale:
        continue
    updates+=1
    if updates==1 or updates%CHECK_EVERY==0 or updates==SUPERLAYER_STEPS:
        current=probe_metrics(superlayer,PROBE_CACHE)
        history.append({"step":updates,**current})
        rank=current["nmse"]+1.5*(1-current["cosine"])
        if rank<best_rank:
            best=current
            best_rank=rank
            best_state=cpu_snapshot(superlayer)
            best_step=updates
        print(f"Superlayer {updates:4d}/{SUPERLAYER_STEPS}: "
              f"probe cosine={current['cosine']:.6f} "
              f"NMSE={current['nmse']:.5f} "
              f"best={best['cosine']:.6f} @ {best_step}")
superlayer.load_state_dict(best_state,strict=True)
superlayer.eval().requires_grad_(False)
print("Probe-selected superlayer:",best,"best step",best_step)
del optimizer,scaler,best_state
torch.cuda.empty_cache()


In [ ]:
#@title 8. Independent blind segment audit and direct CeNN contribution ablation
superlayer.eval()
superlayer.set_cenn_enabled(True)
audit_with=probe_metrics(superlayer,AUDIT_CACHE)
superlayer.set_cenn_enabled(False)
audit_without=probe_metrics(superlayer,AUDIT_CACHE)
superlayer.set_cenn_enabled(True)
cenn_effect={
    "audit_with_cenn":audit_with,
    "audit_without_cenn":audit_without,
    "cosine_gain":audit_with["cosine"]-audit_without["cosine"],
    "nmse_reduction":audit_without["nmse"]-audit_with["nmse"],
    "cenn_gain_stage1":float(torch.sigmoid(superlayer.cenn_gain1).detach()),
    "cenn_gain_stage2":float(torch.sigmoid(superlayer.cenn_gain2).detach()),
    "effective_by_cosine_or_nmse":bool(
        audit_with["cosine"]-audit_without["cosine"]>=0.001 or
        audit_without["nmse"]-audit_with["nmse"]>=0.001),
}
print("BLIND segment output with CeNN:",audit_with)
print("BLIND segment output without CeNN:",audit_without)
print("CeNN causal intervention:",cenn_effect)
print("Stage-A-only trained CeNN was not assumed to help: audit decides.")


In [ ]:
#@title 9. Install actual 15/16/17 -> ONE layer; evaluate downstream DEV
# Clone final stage for benchmark; preserve baseline model originals for rollback.
student_eval.eval().requires_grad_(False)
@torch.inference_mode()
def task_quality(model,items,limit=QUALITY_DEV_LIMIT):
    total=agree=gold_teacher=gold_student=0
    js_total=0.0
    n=min(limit,len(items))
    for i in range(0,n,BATCH):
        chosen=items[i:i+BATCH]
        inp=direct_batch(chosen)
        with torch.autocast("cuda",dtype=amp_dtype):
            teacher_logits,_=teacher(**inp)
            student_logits,_=model(**inp)
        mask=inp["marker_mask"].bool()
        p=torch.softmax(teacher_logits.float().masked_fill(~mask,-1e9),-1)
        q=torch.softmax(student_logits.float().masked_fill(~mask,-1e9),-1)
        gold=torch.tensor([item["gold_index"] for item in chosen],device=device)
        agree+=int((p.argmax(-1)==q.argmax(-1)).sum())
        gold_teacher+=int((p.argmax(-1)==gold).sum())
        gold_student+=int((q.argmax(-1)==gold).sum())
        m=(p+q)*0.5
        js_total+=float((
            0.5*(p*(p.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum(-1)+
            0.5*(q*(q.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum(-1)
        ).sum())
        total+=len(chosen)
    return {"decisions":total,"agreement":agree/max(total,1),
            "gold_teacher":gold_teacher/max(total,1),
            "gold_student":gold_student/max(total,1),
            "mean_js":js_total/max(total,1)}
originals=install_superlayer(student_eval,superlayer)
assert isinstance(student_eval.encoder.layers[16],BypassModernBertLayer)
assert isinstance(student_eval.encoder.layers[17],BypassModernBertLayer)
assert all(type(student_eval.encoder.layers[i])==type(teacher.encoder.layers[i])
           for i in range(len(teacher.encoder.layers)) if i not in (15,16,17))
baseline=task_quality(teacher,DEV_QUALITY)  # Reference-vs-reference, 100%
superlayer.set_cenn_enabled(True)
quality_with=task_quality(student_eval,DEV_QUALITY)
superlayer.set_cenn_enabled(False)
quality_without=task_quality(student_eval,DEV_QUALITY)
superlayer.set_cenn_enabled(True)
print("Native teacher self-check:",baseline)
print("3→1 superlayer WITH CeNN:",quality_with)
print("3→1 superlayer WITHOUT CeNN:",quality_without)
print("The only changed encoder slots are [15,16,17].")


In [ ]:
#@title 10. Measure real CUDA throughput, stage latency, and effective compression
@torch.inference_mode()
def cuda_events_ms(f,warmup=4,repeats=16):
    for _ in range(warmup):f()
    torch.cuda.synchronize()
    times=[]
    for _ in range(repeats):
        a=torch.cuda.Event(enable_timing=True)
        b=torch.cuda.Event(enable_timing=True)
        a.record()
        f()
        b.record()
        b.synchronize()
        times.append(float(a.elapsed_time(b)))
    return {"median_ms":float(np.median(times)),
            "p95_ms":float(np.percentile(times,95)),
            "repeats":repeats}
first=gpu_entry(AUDIT_CACHE[0])
pos=first["position_ids"]
valid=first["valid"]
x=first["x"]
def teacher_segment():
    with torch.autocast("cuda",dtype=amp_dtype):
        state=x
        for idx in (15,16,17):
            state=teacher.encoder.layers[idx](
                state,attention_mask=valid,sliding_window_mask=valid,
                position_ids=pos)[0]
def student_segment():
    with torch.autocast("cuda",dtype=amp_dtype):
        superlayer(x,attention_mask=valid,position_ids=pos)
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    # The 4D native teacher mask can be different from simple 2D padding;
    # benchmark segment with the captured original layer masks instead.
    pass
# Use teacher input/output hooks to capture the ACTUAL kwargs in one direct
# forward so the native segment benchmark is numerically comparable.
reference_call={}
def capture_call(mod,args,kwargs):
    reference_call.update(kwargs)
h=teacher.encoder.layers[15].register_forward_pre_hook(
    capture_call,with_kwargs=True)
try:
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        teacher(**direct_batch(DEV_QUALITY[:BATCH]))
finally:
    h.remove()
def teacher_segment_exact():
    with torch.autocast("cuda",dtype=amp_dtype):
        state=x
        for idx in (15,16,17):
            state=teacher.encoder.layers[idx](state,**reference_call)[0]
# The masks above must match x.shape; use actual AUDIT row forward to get
# exact kwargs for precisely that length rather than a different DEV sample.
def capture_audit_call(mod,args,kwargs):
    reference_call.clear()
    reference_call.update(kwargs)
h=teacher.encoder.layers[15].register_forward_pre_hook(
    capture_audit_call,with_kwargs=True)
try:
    # Generate the original audit batch using the same indexed rows.
    # Cache batch i=0 starts at PROBE_CACHE_BATCHES*BATCH.
    source_batch=batch(dev_items,PROBE_CACHE_BATCHES*BATCH)
    with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
        teacher(**source_batch)
finally:
    h.remove()
assert x.shape[:2]==source_batch["input_ids"].shape
native_stage=cuda_events_ms(teacher_segment_exact)
compressed_stage=cuda_events_ms(student_segment)
bench=[]
for count in (1,BATCH):
    sample=direct_batch(DEV_QUALITY[:count])
    def native_model():
        with torch.autocast("cuda",dtype=amp_dtype):
            teacher(**sample)
    def compressed_model():
        with torch.autocast("cuda",dtype=amp_dtype):
            student_eval(**sample)
    ms_teacher=cuda_events_ms(native_model)
    ms_student=cuda_events_ms(compressed_model)
    bench.append({"batch":count,"seq_len":int(sample["input_ids"].shape[1]),
                  "teacher":ms_teacher,"student":ms_student,
                  "speedup":ms_teacher["median_ms"]/max(ms_student["median_ms"],1e-9)})
print("Native THREE full encoder blocks:",native_stage)
print("New ONE two-window-CeNN block:",compressed_stage)
print("Segment speedup:",native_stage["median_ms"]/
      max(compressed_stage["median_ms"],1e-9))
print("Whole-model measurements:",bench)


In [ ]:
#@title 11. Strict probe/audit/DEV gate, checkpoint, and independent report
local_ok=(audit_with["cosine"]>=TARGET_COSINE and
          audit_with["nmse"]<=TARGET_NMSE)
decision_ok=(quality_with["agreement"]>=TARGET_AGREEMENT and
             quality_with["gold_student"]>=quality_with["gold_teacher"]-MAX_GOLD_DROP)
quality_accepted=bool(local_ok and decision_ok)
stage_speedup=native_stage["median_ms"]/max(compressed_stage["median_ms"],1e-9)
full_speedups=[i["speedup"] for i in bench]
speed_accepted=bool(stage_speedup>1.0 and all(x>1.0 for x in full_speedups))
REPORT={
    "experiment":"Jev_CeNN_two_sparse_windows_three_encoder_layers_to_one",
    "source_teacher":SOURCE_MODEL,"changed_encoder_layers":[15,16,17],
    "active_replacement_blocks":1,"bypassed_complete_blocks":2,
    "other_encoder_layers_unchanged":True,
    "window1":{"radius":LOCAL_RADIUS,"positions":2*LOCAL_RADIUS+1,"stride":1},
    "window2":{"radius":DILATED_RADIUS,"positions":2*DILATED_RADIUS+1,
               "stride":DILATION_STRIDE},
    "max_len":MAX_LEN,"global_two_hop_reachability":True,
    "attention_free":False,"official_test_used":False,"hf_published":False,
    "cenn_stage_a_probe":best_a,"cenn_stage_a_history":history_a,
    "superlayer_stage_b_probe":best,"superlayer_best_step":best_step,
    "superlayer_training_history":history,
    "blind_audit_cenn_on":audit_with,"blind_audit_cenn_off":audit_without,
    "cenn_effect":cenn_effect,
    "downstream_dev_cenn_on":quality_with,
    "downstream_dev_cenn_off":quality_without,
    "native_three_block_cuda":native_stage,
    "new_one_block_cuda":compressed_stage,
    "segment_speedup":stage_speedup,
    "whole_model_cuda":bench,
    "quality_gate_pass":quality_accepted,
    "performance_gate_pass":speed_accepted,
    "deployable_candidate":bool(quality_accepted and speed_accepted),
    "checkpoint_is_experiment_only":True,
}
name="ACCEPTED" if quality_accepted and speed_accepted else "EXPERIMENTAL_NOT_DEPLOYABLE"
weights_file=OUTPUT_DIR/f"jev_cenn_15_16_17_to_1_{name}.pt"
torch.save({
    "architecture":superlayer.architecture,
    "teacher_layers":[15,16,17],
    "window":{"r1":LOCAL_RADIUS,"r2":DILATED_RADIUS,
              "stride":DILATION_STRIDE,"max_len":MAX_LEN},
    "checkpoint_quality_gate_pass":quality_accepted,
    "checkpoint_performance_gate_pass":speed_accepted,
    "student_superlayer_state":cpu_snapshot(superlayer),
    "report":REPORT,
},weights_file)
report_file=OUTPUT_DIR/"jev_cenn_15to17_two_window_report.json"
report_file.write_text(json.dumps(REPORT,indent=2,allow_nan=False))
print("\nCHECKPOINT:",weights_file)
print("REPORT:",report_file)
print("Quality gate:", "PASS" if quality_accepted else "NOT REACHED")
print("GPU speed gate:", "PASS" if speed_accepted else "NOT REACHED")
print("CeNN evidence:",cenn_effect["effective_by_cosine_or_nmse"])
print("Note: even if the quality gate passes, do NOT claim convergence of all")
print("28 blocks or publish this experimental 3-to-1 architecture automatically.")


In [ ]:
#@title 12. Plot CeNN pretraining, superlayer convergence, and measured GPU latency
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
axes[0].plot([h["step"] for h in history_a],
             [h["cosine"] for h in history_a],marker="o")
axes[0].set(xlabel="Stage A CeNN updates",ylabel="PROBE cosine",
            title="Pure CeNN first-layer residual")
axes[1].plot([h["step"] for h in history],
             [h["cosine"] for h in history],marker="o",label="Superlayer")
axes[1].axhline(TARGET_COSINE,ls="--",color="gray",label="Goal 0.99")
axes[1].set(xlabel="Stage B optimizer updates",ylabel="PROBE cosine",
            title="Three teacher blocks → one new layer")
axes[1].legend()
axes[2].bar(["Teacher 15–17","Jev_CeNN ONE"],
            [native_stage["median_ms"],compressed_stage["median_ms"]])
axes[2].set(ylabel="CUDA median ms",title="Matched segment latency")
for ax in axes:ax.grid(axis="y",alpha=.2)
fig.tight_layout()
fig_path=OUTPUT_DIR/"jev_cenn_15to17_two_window.png"
fig.savefig(fig_path,dpi=160,bbox_inches="tight")
plt.show()
print("Figure:",fig_path)
